# 第 13 章 繰り返しゲームと屈折需要曲線: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/micro-basics-programming/13-repeated-games-and-kinked-demand.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
temptation = 5    # T: 自分だけ裏切る（T > R > P > S）
reward = 3        # R: 2 人とも協調する
punishment = 1    # P: 2 人とも裏切る
sucker = 0        # S: 自分だけ協調する
delta = 0.6       # 割引因子（将来を、どれだけ大切にするか）
rounds = 4        # 繰り返す回数
kink_quantity = 40   # 屈折需要: 屈折点の数量
kink_price = 50      # 屈折点の価格
upper_slope = 0.5    # 屈折点より上（価格が高い側）の傾き
lower_slope = 2      # 屈折点より下の傾き
marginal_cost = 30   # 限界費用（価格が動くか、確かめる）
cost_low = 10        # 図に描く、低い限界費用
cost_high = 25       # 図に描く、高い限界費用
q_max = 100          # 屈折需要の図の数量の範囲
p_max = 80           # 屈折需要の図の価格の範囲

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 繰り返しゲーム

In [ ]:
from lib.chapter12.game import prisoners_dilemma
from lib.chapter13.repeated import (
    always_defect,
    cooperation_threshold,
    cooperation_value,
    deviation_value,
    play,
    tit_for_tat,
    total_payoffs,
    trigger_sustains_cooperation,
)

game = prisoners_dilemma(temptation, reward, punishment, sucker)
threshold = cooperation_threshold(temptation, reward, punishment)
print(f"協調が続く、割引因子の閾値: {threshold:.4f}")
sustains = trigger_sustains_cooperation(temptation, reward, punishment, delta)
print(f"δ = {delta}: 協調 {cooperation_value(reward, delta):.4f}、"
      f"裏切り {deviation_value(temptation, punishment, delta):.4f}、"
      f"協調が続くか {sustains}")
history = play(game, tit_for_tat, always_defect, rounds)
print("しっぺ返し対いつも裏切り:", total_payoffs(game, history))
history = play(game, tit_for_tat, tit_for_tat, rounds)
print("しっぺ返し同士:", total_payoffs(game, history))

## 屈折需要曲線

In [ ]:
from lib.chapter13.kinked import (
    KinkedDemand,
    marginal_revenue_above,
    marginal_revenue_below,
    marginal_revenue_gap,
    price_stays,
)

kinked = KinkedDemand(kink_quantity, kink_price, upper_slope, lower_slope)
print(f"限界収入: 上 {marginal_revenue_above(kinked):.2f}、"
      f"下 {marginal_revenue_below(kinked):.2f}、"
      f"途切れ {marginal_revenue_gap(kinked):.2f}")
print(f"限界費用 {marginal_cost} で、価格は動かないか: "
      f"{price_stays(kinked, marginal_cost)}")

## 図

In [ ]:
from lib.chapter13.plot import plot_trigger

plot_trigger(temptation, reward, punishment, 0.95, 100, 60)

In [ ]:
from lib.chapter13.plot import plot_kinked

plot_kinked(kinked, q_max, p_max, cost_low, cost_high)

## 試してみよう

- `temptation = 6` にすると、閾値は 0.6 に上がります。`delta = 0.5` では、協調は
  続かず（False）、`delta = 0.6` なら続きます（True）。裏切りの誘惑が大きいほど、
  将来を、より大切にしないと、協調は続きません
- `punishment = 2` にすると、閾値は約 0.667 に上がります。罰（P）が軽いと、
  裏切っても失うものが小さいので、協調が続きにくくなります
- `rounds = 10` にすると、しっぺ返し対いつも裏切りは (9, 14)、しっぺ返し同士は
  (30, 30) です。最初の 1 回の搾取は、そのままで、回数が増えるほど、協調の差が開きます
- `lower_slope = 1` にすると、途切れは 20.0 に縮みます（2 のときは 60.0、4 では
  140.0）。屈折が弱いほど、価格が動かない範囲も、狭くなります
- `upper_slope = 1` にすると、`marginal_cost = 30` では、価格は動きます（False）。
  途切れの範囲が、10 から −30 の間に、変わるためです